# LR1. Ochistka (02_cleaning)
Kazhdyy etap — otdelnaya yacheyka: poyasnenie -> vyzov funktsii iz src.cleaning -> display bylo/stalo.
Ispolnyaetsya tselikom posle Restart & Run All.

In [ ]:
import pandas as pd
from IPython.display import display
from src.cleaning import *
SRC = r'C:\Users\artur\Desktop\датасеты\Земский\csv\1_датасет_население_возраст_и_пол.csv'
df = pd.read_csv(SRC, sep=';', encoding='utf-8-sig', low_memory=False)
print('start:', df.shape)

### 1. Pereimenovanie stolbtsov (ru -> snake_case)

In [ ]:
before=len(df); df=rename_columns(df)
print(f'bylo {before}, stalo {len(df)}')
display(df.columns.tolist())

### 2. Yavnoe privedenie tipov (zapayataya -> tochka, oshibki -> NaN)

In [ ]:
before=len(df); df=coerce_types(df)
print(f'bylo {before}, stalo {len(df)}')
display(df[['year','value']].dtypes.to_frame('tip'))
print('NaN v value:', int(df['value'].isna().sum()))

### 3. Skrytye propuski -> NaN

In [ ]:
before=len(df); df=normalize_hidden_missing(df)
print(f'bylo {before}, stalo {len(df)}')

### 4. Udalenie sluzhebnogo stolbtsa footnote

In [ ]:
before=df.shape; df=drop_footnote(df)
print(f'bylo {before}, stalo {df.shape}')

### 5. Polnye dubli + dubli po klyuchu (country, indicator, year)

In [ ]:
before=len(df); df=drop_full_duplicates(df); df=drop_key_duplicates(df)
print(f'bylo {before}, stalo {len(df)}')

### 6. Normalizatsiya kategoriy (strip, redkie -> OTHER)

In [ ]:
before=len(df); df=normalize_categories(df)
print(f'bylo {before}, stalo {len(df)}')
display(df['region'].value_counts(dropna=False).head(10))

### 7. Tekst indicator_def (html, probely)

In [ ]:
before=len(df); df=clean_text_field(df)
print(f'bylo {before}, stalo {len(df)}')

### 8. Biznes-pravila (flagi, bez udaleniya)

In [ ]:
before=len(df); df=check_business_rules(df)
print(f'bylo {before}, stalo {len(df)}')
print('bad_year:', int(df['flag_bad_year'].sum()), 'bad_value:', int(df['flag_bad_value'].sum()))

### 9. Vybrosy dvumya sposobami (IQR + 3 sigma, flagi)

In [ ]:
before=len(df); df=flag_outliers(df)
print(f'bylo {before}, stalo {len(df)}')
print('iqr:', int(df['flag_out_iqr'].sum()), 'sigma:', int(df['flag_out_sigma'].sum()))

### 10. Optimizirovannaya skhema + sokhranenie

In [ ]:
before=len(df); df=apply_optimized_schema(df)
print(f'bylo {before}, stalo {len(df)}')
df.info(memory_usage='deep')
df.to_parquet('data/processed/clean.parquet', compression='zstd')
print('saved: data/processed/clean.parquet', df.shape)

### Proverka: soglasovannost s pipeline (`python -m src.pipeline clean`) — sverka hash; idempotentnost — povtor ne menyaet rezultat.